In [ ]:
import pandas as pd
import numpy as np
import glob
import tqdm
from imblearn.under_sampling import TomekLinks
from imblearn.over_sampling import SMOTE


In [ ]:
class CICIDSDataGenerator:
    def __init__(self, root="data/intrusion_data", n_limit=100000):
        self.ROOT = root
        self.n_limit = n_limit
        self.df_org = None
        self.df = None
        self.labels = None
        self.drop_cols = ['id','Flow ID','Src IP','Src Port','Dst IP','Dst Port','Protocol','Timestamp','Attempted Category']
        self.string_cols = None

    def load_data(self):
        df_list = []
        for path in tqdm.tqdm(glob.glob(f"{self.ROOT}/*"), desc="Loading Data"):
            df_list.append(pd.read_csv(path))
        self.df_org = pd.concat(df_list, ignore_index=True)
        del df_list
        print(f"Loaded {len(self.df_org)} rows")

    def normalize_labels(self):
        def normalize_label(label):
            label = str(label)
            if label.endswith(' - Attempted'):
                label = label.replace(' - Attempted', '')
            if 'Web Attack - Brute Force' in label:
                return None
            if label.startswith('BENIGN'): 
                return 'BENIGN'
            if 'DoS' in label: 
                return 'DoS'
            if 'DDoS' in label: 
                return 'DDoS'
            if 'FTP' in label or 'SSH' in label: 
                return 'BruteForce'
            if 'Web Attack' in label:
                if 'XSS' in label: 
                    return 'XSS Attack'
                if 'SQL Injection' in label: 
                    return 'SQL Injection'
                return 'WebAttack'
            if 'Infiltration' in label: 
                return 'Infiltration'
            if 'Botnet' in label: 
                return 'Botnet'
            if 'Heartbleed' in label: 
                return None
            return label

        self.df_org['Label'] = self.df_org['Label'].map(normalize_label)
        self.df_org = self.df_org[self.df_org['Label'].notna()]
        print("Labels normalized")

    def random_undersample(self):
        groups = []
        print(f"Before Random Undersampling: {len(self.df_org)} rows")
        for label, group in self.df_org.groupby('Label'):
            if len(group) > self.n_limit:
                group = group.sample(n=self.n_limit, random_state=42)
            groups.append(group)
        self.df_org = pd.concat(groups, axis=0).reset_index(drop=True)
        del groups
        print(f"After Random Undersampling: {len(self.df_org)} rows")

    def prepare_numeric_data(self):
        self.string_cols = self.df_org[self.drop_cols].reset_index(drop=True)
        self.df = self.df_org.drop(columns=self.drop_cols + ['Label']).reset_index(drop=True)
        self.labels = self.df_org['Label'].astype('category').reset_index(drop=True)
        self.df.replace([np.inf, -np.inf], np.nan, inplace=True)
        self.df.dropna(axis=0, inplace=True)
        self.labels = self.labels[self.df.index]
        print(f"Prepared numeric features: {self.df.shape}")

    def tomek_undersample(self):
        tl = TomekLinks(n_jobs=-1)
        X_res, y_res = tl.fit_resample(self.df, self.labels)
        self.df = pd.DataFrame(X_res, columns=self.df.columns)
        self.labels = pd.Series(y_res).reset_index(drop=True)
        print(f"After Tomek links: {self.df.shape[0]} rows")

    def smote_oversample(self):
        smote = SMOTE(sampling_strategy='auto', random_state=42)
        X_res, y_res = smote.fit_resample(generator.df, generator.labels)
        generator.df = pd.DataFrame(X_res, columns=generator.df.columns)
        generator.labels = pd.Series(y_res).reset_index(drop=True)
        print(f"After SMOTE: {generator.df.shape[0]} rows")


    def add_back_columns(self):
        n_synthetic = len(self.df) - len(self.string_cols)
        if n_synthetic > 0:
            random_strings = {
                col: np.random.choice(self.string_cols[col], size=n_synthetic, replace=True)
                for col in self.drop_cols
            }
            df_strings_synthetic = pd.DataFrame(random_strings)
            df_strings_full = pd.concat([self.string_cols, df_strings_synthetic], axis=0).reset_index(drop=True)
        else:
            df_strings_full = self.string_cols.reset_index(drop=True)

        df_final = pd.concat([df_strings_full, self.df.reset_index(drop=True), self.labels.rename('Label')], axis=1)
        return df_final

    def save(self, path="data/cicids_final.csv"):
        df_final = self.add_back_columns()
        df_final.to_csv(path, index=False)
        print(f"Saved final DataFrame: {df_final.shape[0]} rows to {path}")


generator = CICIDSDataGenerator(n_limit=100000)

generator.load_data()           
generator.normalize_labels()     
generator.random_undersample()   
generator.prepare_numeric_data() 
generator.tomek_undersample()    
generator.smote_oversample()    
generator.save("data/cicids_cleaned_resampled.csv") 